# Fairness checks 

A learning and verification notebook for small synthetic datasets. Results are checked against hand calculations and Fairlearn.

**Workflow:** prepare data, compute per-group metrics, compare fairness gaps, assess AUROC, use the shared implementation, and check edge cases.

Run all cells from top to bottom with the project's `.venv` kernel. These examples are not an audit report: groups are small, and suppression, significance tests, and confidence intervals are not applied here.


## 1. Environment setup

Import the shared functions from `fairness/metrics.py` instead of redefining them in the notebook.
The setup supports running from either the project root or the `notebooks` directory.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "fairness" / "metrics.py").is_file():
    raise RuntimeError("Run this notebook from the project root or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from fairlearn.metrics import (
    MetricFrame,
    count,
    demographic_parity_difference,
    equalized_odds_difference,
    false_positive_rate,
    selection_rate,
    true_positive_rate,
)
from sklearn.metrics import precision_score, roc_auc_score

from fairness.metrics import base_rate, compute_group_metrics

print("Python:", sys.executable)


Python: C:\Users\A TO Z\projects\healthequity-agent\.venv\Scripts\python.exe


## 2. Toy dataset

Each row represents one observation. `outcome` is the ground truth, `prediction` is the model's predicted label, and `group` identifies the comparison group.
The value 1 represents the positive class. Each group contains four observations, including two actual positives.


In [2]:
toy = pd.DataFrame({
    "outcome": [1, 1, 0, 0, 1, 1, 0, 0],
    "prediction": [1, 0, 1, 0, 1, 1, 1, 0],
    "group": ["A", "A", "A", "A", "B", "B", "B", "B"],
})

toy


,outcome,prediction,group
0,1,1,A
1,1,0,A
2,0,1,A
3,0,0,A
4,1,1,B
5,1,1,B
6,0,1,B
7,0,0,B


## 3. Per-group metrics

`MetricFrame` splits observations by `sensitive_features` and applies each metric to the ground truth and predictions within each group.

| Metric | Calculation | A | B |
|---|---|---:|---:|
| Count | Number of observations | 4 | 4 |
| Base rate | Actual positives / observations | 2/4 | 2/4 |
| Selection rate | Predicted positives / observations | 2/4 | 3/4 |

For binary labels, the mean of `outcome` equals the fraction of actual positives. This is what `base_rate` computes.


In [3]:
basic_frame = MetricFrame(
    metrics={
        "count": count,
        "base_rate": base_rate,
        "selection_rate": selection_rate,
    },
    y_true=toy["outcome"],
    y_pred=toy["prediction"],
    sensitive_features=toy["group"],
)

basic_frame.by_group


,count,base_rate,selection_rate
group,,,
A,4.0,0.5,0.50
B,4.0,0.5,0.75


### TPR, FPR and PPV

| Group | TP | FN | FP | TN |
|---|---:|---:|---:|---:|
| A | 1 | 1 | 1 | 1 |
| B | 2 | 0 | 1 | 1 |

- **TPR = TP / (TP + FN):** What fraction of actual positives were detected? A = 0.5, B = 1.
- **FPR = FP / (FP + TN):** What fraction of actual negatives were incorrectly flagged? A = B = 0.5.
- **PPV = TP / (TP + FP):** What fraction of positive predictions were correct? A = 0.5, B = 2/3.

The library functions are used directly here because all denominators are positive in this example. The shared implementation in `fairness/metrics.py` returns unavailable values for zero denominators, as illustrated in the edge-case section.


In [4]:
frame = MetricFrame(
    metrics={
        "count": count,
        "base_rate": base_rate,
        "selection_rate": selection_rate,
        "tpr": true_positive_rate,
        "fpr": false_positive_rate,
        "ppv": precision_score,
    },
    y_true=toy["outcome"],
    y_pred=toy["prediction"],
    sensitive_features=toy["group"],
)

frame.by_group


,count,base_rate,selection_rate,tpr,fpr,ppv
group,,,,,,
A,4.0,0.5,0.50,0.5,0.5,0.500000
B,4.0,0.5,0.75,1.0,0.5,0.666667


## 4. Fairness gaps

- **Demographic parity difference:** Maximum selection rate minus minimum selection rate = 0.25.
- **Equalized odds difference:** The larger of the TPR gap and FPR gap = max(0.5, 0) = 0.5.
- **PPV difference:** Maximum PPV minus minimum PPV = 2/3 - 1/2, approximately 0.166667.

A gap of 0.25 means **25 percentage points**. A zero gap indicates equality only for the metric being compared.

**Example limitations:** Both groups are present and every metric is defined. Do not generalize this `max - min` calculation to results containing `NaN`: Pandas skips missing values by default and may produce a misleading gap. `compute_fairness_gaps` preserves unavailable comparisons with explicit reasons. Reportable comparisons require eligibility filtering with `fairness/policy.py`; the tiny groups here illustrate arithmetic without applying the reporting policy.


In [5]:
gaps = frame.by_group.max() - frame.by_group.min()

fairness_gaps = pd.Series({
    "demographic_parity_difference": gaps["selection_rate"],
    "tpr_gap": gaps["tpr"],
    "fpr_gap": gaps["fpr"],
    "equalized_odds_difference": max(gaps["tpr"], gaps["fpr"]),
    "ppv_difference": gaps["ppv"],
})

fairness_gaps


demographic_parity_difference    0.250000
tpr_gap                          0.500000
fpr_gap                          0.000000
equalized_odds_difference        0.500000
ppv_difference                   0.166667
dtype: float64

### Reference checks

Compare DPD and EOD with Fairlearn's reference functions and all three headline gaps with hand calculations, using a numerical tolerance below `1e-6`.
These notebook checks support learning; automated tests for the shared functions are also available in `test/test_metrics.py`.


In [6]:
dpd = demographic_parity_difference(
    y_true=toy["outcome"],
    y_pred=toy["prediction"],
    sensitive_features=toy["group"],
)

eod = equalized_odds_difference(
    y_true=toy["outcome"],
    y_pred=toy["prediction"],
    sensitive_features=toy["group"],
)

print("Fairlearn DPD:", dpd)
print("Fairlearn EOD:", eod)

assert abs(
    dpd - fairness_gaps["demographic_parity_difference"]
) < 1e-6

assert abs(
    eod - fairness_gaps["equalized_odds_difference"]
) < 1e-6

assert abs(dpd - 0.25) < 1e-6
assert abs(eod - 0.5) < 1e-6
assert abs(fairness_gaps["ppv_difference"] - (2 / 3 - 1 / 2)) < 1e-6

print("Fairlearn and hand-calculation checks passed")


Fairlearn DPD: 0.25
Fairlearn EOD: 0.5
Fairlearn and hand-calculation checks passed


## 5. AUROC and probability scores

AUROC uses `score` instead of the final `prediction` label. It measures how well actual positives rank above actual negatives.
The following scores are synthetic and agree with the example's predicted labels at a threshold of 0.5.

| Group | Scores of actual positives | Scores of actual negatives | Correctly ranked pairs | AUROC |
|---|---|---|---|---:|
| A | 0.9, 0.4 | 0.7, 0.2 | 3 of 4 | 0.75 |
| B | 0.8, 0.7 | 0.6, 0.1 | 4 of 4 | 1.00 |

A tied pair receives half a point. Perfect AUROC on this tiny example does not establish perfect performance on new data.


In [7]:
toy["score"] = [
    0.9, 0.4, 0.7, 0.2,  # A
    0.8, 0.7, 0.6, 0.1,  # B
]

auc_frame = MetricFrame(
    metrics={"auroc": roc_auc_score},
    y_true=toy["outcome"],
    y_pred=toy["score"],
    sensitive_features=toy["group"],
)

auc_frame.by_group


,auroc
group,
A,0.75
B,1.00


In [8]:
assert abs(auc_frame.by_group.loc["A", "auroc"] - 0.75) < 1e-6
assert abs(auc_frame.by_group.loc["B", "auroc"] - 1.0) < 1e-6
print("AUROC hand-calculation checks passed")


AUROC hand-calculation checks passed


## 6. Shared implementation

`compute_group_metrics` combines the metrics, checks the data with the project's validator, and returns `NaN` for undefined metrics. Rows are not dropped, and the function is imported rather than redefined in this notebook.

These are raw computed results. `compute_fairness_gaps` supplies reasons for unavailable rates and gaps; `fairness/policy.py` supplies group eligibility, filtering, and severity classification. This notebook illustrates metric calculations without presenting its tiny groups as reportable audit findings.


In [9]:
group_metrics = compute_group_metrics(
    toy,
    group_column="group",
    score_column="score",
)

group_metrics


,count,base_rate,selection_rate,tpr,fpr,ppv,auroc
group,,,,,,,
A,4.0,0.5,0.50,0.5,0.5,0.500000,0.75
B,4.0,0.5,0.75,1.0,0.5,0.666667,1.00


### Optional score column

When `score_column` is omitted, only label-based metrics are computed, and the AUROC column is absent.


In [10]:
without_score = compute_group_metrics(
    toy.drop(columns="score"),
    group_column="group",
)

assert "auroc" not in without_score.columns
pd.testing.assert_frame_equal(
    without_score,
    group_metrics.drop(columns="auroc"),
)
without_score


,count,base_rate,selection_rate,tpr,fpr,ppv
group,,,,,,
A,4.0,0.5,0.50,0.5,0.5,0.500000
B,4.0,0.5,0.75,1.0,0.5,0.666667


## 7. Undefined metrics and edge cases

**Zero is a computed value; `NaN` means the metric is undefined.**

| Metric | Condition making it undefined |
|---|---|
| TPR | No actual positives: TP + FN = 0 |
| FPR | No actual negatives: FP + TN = 0 |
| PPV | No positive predictions: TP + FP = 0 |
| AUROC | Ground truth contains only one class |

### 7.1 No positive predictions

PPV is undefined, while TPR and FPR are zero. AUROC is 1 because the score ranking is correct; all scores fall below the decision threshold of 0.5.


In [11]:
edge_case = pd.DataFrame({
    "outcome": [1, 1, 0, 0],
    "prediction": [0, 0, 0, 0],
    "group": ["A"] * 4,
    "score": [0.4, 0.3, 0.2, 0.1],
})

no_predictions_result = compute_group_metrics(edge_case, "group", "score")
assert pd.isna(no_predictions_result.loc["A", "ppv"])
assert no_predictions_result.loc["A", "tpr"] == 0.0
assert no_predictions_result.loc["A", "fpr"] == 0.0
assert no_predictions_result.loc["A", "auroc"] == 1.0
print("No-positive-predictions checks passed")
no_predictions_result


No-positive-predictions checks passed


,count,base_rate,selection_rate,tpr,fpr,ppv,auroc
group,,,,,,,
A,4.0,0.5,0.0,0.0,0.0,NaN,1.0


### 7.2 No positive outcomes

TPR and AUROC are undefined. FPR = 2/4 = 0.5, and PPV = 0/2 = 0 because both positive predictions are incorrect.


In [12]:
no_positives = pd.DataFrame({
    "outcome": [0, 0, 0, 0],
    "prediction": [1, 0, 1, 0],
    "group": ["A"] * 4,
    "score": [0.8, 0.2, 0.6, 0.1],
})

no_positives_result = compute_group_metrics(no_positives, "group", "score")
assert pd.isna(no_positives_result.loc["A", "tpr"])
assert pd.isna(no_positives_result.loc["A", "auroc"])
assert no_positives_result.loc["A", "fpr"] == 0.5
assert no_positives_result.loc["A", "ppv"] == 0.0
print("No-positive-outcomes checks passed")
no_positives_result


No-positive-outcomes checks passed


,count,base_rate,selection_rate,tpr,fpr,ppv,auroc
group,,,,,,,
A,4.0,0.0,0.5,NaN,0.5,0.0,NaN


### 7.3 No negative outcomes

FPR and AUROC are undefined. TPR = 2/4 = 0.5, and PPV = 2/2 = 1: both positive predictions are correct, but the model misses two actual positives.


In [13]:
no_negatives = pd.DataFrame({
    "outcome": [1, 1, 1, 1],
    "prediction": [1, 0, 1, 0],
    "group": ["A"] * 4,
    "score": [0.8, 0.2, 0.6, 0.1],
})

no_negatives_result = compute_group_metrics(no_negatives, "group", "score")
assert no_negatives_result.loc["A", "base_rate"] == 1.0
assert no_negatives_result.loc["A", "tpr"] == 0.5
assert pd.isna(no_negatives_result.loc["A", "fpr"])
assert no_negatives_result.loc["A", "ppv"] == 1.0
assert pd.isna(no_negatives_result.loc["A", "auroc"])
print("No-negative-outcomes checks passed")
no_negatives_result


No-negative-outcomes checks passed


,count,base_rate,selection_rate,tpr,fpr,ppv,auroc
group,,,,,,,
A,4.0,1.0,0.5,0.5,NaN,1.0,NaN


## Scope and interpretation

This notebook verifies metric arithmetic, comparisons with Fairlearn, optional AUROC, and undefined-metric behavior using small synthetic datasets. These examples do not establish fairness of a clinical model.

The shared modules separate metric computation (`fairness/metrics.py`), statistical tests and intervals (`fairness/stats.py`), reporting policies (`fairness/policy.py`), and synthetic data generation (`fairness/scenarios.py`). [Methodology](../docs/methodology.md) defines the assumptions and interpretation rules. Eligibility filtering is required before reportable comparisons; a zero gap is distinct from an unavailable comparison.

Shared-function tests are in `test/test_metrics.py`. Run them from the project root:

```powershell
.\.venv\Scripts\python.exe -m pytest test/test_metrics.py -q
```
